In [11]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

# --------------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------------
IN_DIR = Path(r"C:\Drought\Regridding and data clipping\GLEAM")
E_FILE  = IN_DIR / "GLEAM_E_India_0p05deg_2000-2023.nc"
EP_FILE = IN_DIR / "GLEAM_Ep_India_0p05deg_2000-2023.nc"

OUT_FILE = IN_DIR / "GLEAM_ET_PET_ESI_monthly_India_0p05deg_2000present.nc"

BASE_START = "2001-01"
BASE_END   = "2020-12"
CHUNKS = {"time": 36, "lat": 300, "lon": 300}

# --------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------
def to_month_end(da):
    """Ensure timestamps are month-end, sorted, unique."""
    t = pd.to_datetime(da["time"].values)
    mend = pd.PeriodIndex(t, freq="M").to_timestamp("M")
    da2 = da.assign_coords(time=mend)
    # drop duplicate stamps if any
    _, idx = np.unique(da2["time"].values, return_index=True)
    return da2.isel(time=np.sort(idx)).sortby("time")

def seconds_per_month(time_index):
    t = pd.to_datetime(time_index)
    days = t.to_period("M").days_in_month.values
    return (days * 24 * 3600).astype("float64")

def to_mm_per_month(var):
    """
    Convert a water flux/depth to mm/month.
    Accepts common unit forms:
      - 'mm month-1' or 'mm/month'  -> as-is
      - 'mm day-1'  or 'mm/day'     -> multiply by days in month
      - 'kg m-2 s-1'                -> multiply by seconds per month (1 kg m-2 == 1 mm)
      - 'kg m-2' (monthly)          -> as-is, set units to mm/month
    If units missing/unknown, assume already mm/month.
    """
    units = str(getattr(var, "units", "")).lower().replace("**", "^").strip()
    t = var["time"].values
    if "mm month" in units or "mm/month" in units:
        return var
    if "mm day" in units or "mm/day" in units:
        days = xr.DataArray(pd.to_datetime(t).to_period("M").days_in_month.values,
                            coords={"time": var["time"]}, dims=("time",))
        out = var * days
        return out.assign_attrs(units="mm month-1")
    if "kg" in units and "s-1" in units:
        spm = xr.DataArray(seconds_per_month(t), coords={"time": var["time"]}, dims=("time",))
        out = var * spm
        return out.assign_attrs(units="mm month-1")
    if units == "kg m-2" or ("kg" in units and "s-1" not in units):
        return var.assign_attrs(units="mm month-1")
    # default
    return var.assign_attrs(units="mm month-1")

def monthwise_z(da, base_start, base_end, clip_to=None):
    """
    Calendar-month Z-scores: Z = (x - μ_m) / σ_m  per cell and calendar month.
    """
    base = da.sel(time=slice(base_start, base_end))
    outs = []
    for m in range(1, 13):
        xm = da.where(da["time.month"] == m, drop=True)
        bm = base.where(base["time.month"] == m, drop=True)
        mu = bm.mean("time", skipna=True)
        sd = bm.std("time", ddof=1, skipna=True)
        zm = (xm - mu) / sd
        outs.append(zm)
    out = xr.concat(outs, dim="time").sortby("time")
    if clip_to is not None:
        out = out.clip(clip_to[0], clip_to[1])
    return out

# --------------------------------------------------------------------
# Load GLEAM E and Ep
# --------------------------------------------------------------------
print("Loading GLEAM E (ET) …")
ds_e = xr.open_dataset(E_FILE)
assert "E" in ds_e, "Variable 'E' not found in " + str(E_FILE)
E = to_month_end(ds_e["E"])

print("Loading GLEAM Ep (PET) …")
ds_ep = xr.open_dataset(EP_FILE)
assert "Ep" in ds_ep, "Variable 'Ep' not found in " + str(EP_FILE)
Ep = to_month_end(ds_ep["Ep"])

# Convert to mm/month (GLEAM monthly is usually already mm/month; this is safe)
ET_mm  = to_mm_per_month(E).rename("ET_mm").assign_attrs(long_name="GLEAM actual ET (monthly total)")
PET_mm = to_mm_per_month(Ep).rename("PET_mm").assign_attrs(long_name="GLEAM potential ET (monthly total)")

# Align time
common_time = np.intersect1d(ET_mm["time"].values, PET_mm["time"].values)
ET_mm  = ET_mm.sel(time=common_time).chunk(CHUNKS)
PET_mm = PET_mm.sel(time=common_time).chunk(CHUNKS)

# --------------------------------------------------------------------
# Derived metrics
# --------------------------------------------------------------------
# Safe ET/PET ratio (avoid division by ~0)
eps = 1e-6
ET_to_PET = xr.where(PET_mm > eps, ET_mm / PET_mm, np.nan).rename("ET_to_PET").assign_attrs(units="1", long_name="ET/PET ratio")

# Z-scores per calendar month (2001–2020)
ET_z  = monthwise_z(ET_mm,  BASE_START, BASE_END, clip_to=(-3, 3)).rename("ET_z").assign_attrs(units="1", long_name="Z(ET)")
PET_z = monthwise_z(PET_mm, BASE_START, BASE_END, clip_to=(-3, 3)).rename("PET_z").assign_attrs(units="1", long_name="Z(PET)")

# ESI proxy: Z(ET) − Z(PET)  (more negative => drier)
ESI_proxy = (ET_z - PET_z).rename("ESI_proxy").assign_attrs(units="1", long_name="ESI proxy = Z(ET) − Z(PET)")

# Optional: Z of ET/PET itself
ET_to_PET_z = monthwise_z(ET_to_PET, BASE_START, BASE_END, clip_to=(-3, 3)).rename("ET_to_PET_z").assign_attrs(units="1")

# --------------------------------------------------------------------
# Save NetCDF
# --------------------------------------------------------------------
print(f"Writing {OUT_FILE} …")
ds_out = xr.Dataset(
    data_vars=dict(
        ET_mm=ET_mm,
        PET_mm=PET_mm,
        ET_to_PET=ET_to_PET,
        ET_z=ET_z,
        PET_z=PET_z,
        ESI_proxy=ESI_proxy,
        ET_to_PET_z=ET_to_PET_z,
    ),
    attrs=dict(
        title="GLEAM ET, PET, ET/PET and ESI proxy for India (0.05°)",
        summary="Derived from GLEAM monthly E and Ep; calendar-month Z-scores vs 2001–2020; "
                "ESI proxy = Z(ET) − Z(PET).",
        conventions="CF-1.8",
        crs="EPSG:4326",
        spatial_resolution="0.05 degree",
        temporal_aggregation="monthly",
        history="Generated by make_esi_from_gleam.py",
        baseline=f"{BASE_START}..{BASE_END}",
        notes="Unit conversion heuristic: if units unknown, assumed mm/month.",
    ),
)
comp = dict(zlib=True, complevel=4)
encoding = {v: comp for v in ds_out.data_vars}
encoding.update({"lat": {"dtype": "float32"}, "lon": {"dtype": "float32"}})
ds_out.to_netcdf(OUT_FILE, encoding=encoding)
print("Done.")


Loading GLEAM E (ET) …
Loading GLEAM Ep (PET) …
Writing C:\Drought\Regridding and data clipping\GLEAM\GLEAM_ET_PET_ESI_monthly_India_0p05deg_2000present.nc …


C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\dask\array\numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\dask\core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\dask\core.py:127: RuntimeWarning: divide by zero encountered in divide
  return func(*(_execute_task(a, cache) for a in args))


Done.
